# Microbiome Analysis and Visualization with phyloseq

These instructions guide you through the analysis and visualization of microbiome data
using the **phyloseq** package in R. We will load data that was processed in QIIME2,
create plots of taxonomic composition, and calculate alpha and beta diversity.

Most of these instructions are modified from the
[Denef lab howto](http://deneflab.github.io/MicrobeMiseq/demos/mothur_2_phyloseq.html).

### Learning objectives

By the end of this lab, you should be able to:

1. Load a phyloseq object and identify its parts (feature table, taxonomy, tree, sample metadata).
2. Convert read counts to relative abundance and build taxonomic composition bar plots.
3. Visualize differences in community composition (beta diversity) with abundance-based and phylogenetic distances, and test those differences statistically.
4. Estimate alpha diversity with repeated rarefaction and compare it between groups.
5. Interpret what microbial signal in negative controls means for a sequencing study.

### About the data

Every sample in this dataset is a **negative control** from a 16S rRNA sequencing study.
There are two kinds (the `Type` column):

* **PCR** negatives: PCR reactions with no DNA template added.
* **DNA** negatives: blank DNA extractions, run through the extraction kit with no sample.

In a perfect world, these would have zero reads. As you work through the lab, keep asking yourself:
*Where did these microbes come from, and what do they mean for the real samples sequenced alongside them?*

---

### Before you start

1. **Save your own copy.** Go to **File → Save a copy in Drive**. Work in that copy, or your edits and notes will be lost when you close the tab.
2. **Check the runtime.** This notebook runs **R**, not Python. Under **Runtime → Change runtime type**, the language should say **R**.
3. **Run cells in order**, top to bottom. Click a code cell and press **Shift + Enter** to run it.
4. If you step away for a while, Colab may reset your session. If you get errors like *could not find function* or *object not found*, rerun the cells from the top.
5. Cells marked **✏️ Your turn** are for you to write or edit code. Cells marked **💭 Stop and think** ask you to write an answer in the text cell below them.

# Section 1: Getting your workspace ready

### Install the packages

Colab starts with a fresh computer every session, so we need to install the packages we use.
The cell below sets up pre-built (binary) R packages, which install in a minute or two
instead of the 10+ minutes it takes to build them from scratch.

**Run this cell once at the start of each session and wait for it to finish** before moving on.
If the packages are already installed, it skips the installation.

In [ ]:
# Install packages (once per session) -------------------------------------
if (!requireNamespace("phyloseq", quietly = TRUE) ||
    !requireNamespace("colorspace", quietly = TRUE)) {
  # Set up r2u, a source of pre-built R packages for Ubuntu (the system Colab runs on)
  codename <- system("lsb_release -cs", intern = TRUE)
  download.file(paste0("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_",
                       codename, ".sh"), "add_cranapt.sh")
  Sys.chmod("add_cranapt.sh", "0755")
  system("./add_cranapt.sh")
  bspm::enable()
  options(bspm.version.check = FALSE)

  install.packages(c("phyloseq", "colorspace"))
}
cat("Setup complete.\n")

### Load the packages

Each package has a specific role:
* `phyloseq`: The core package for importing, storing, and analyzing microbiome data.
* `ggplot2`: A powerful tool for creating plots.
* `dplyr`: Tools for filtering and sorting data, including the `%>%` pipe.
* `colorspace`: Color palettes for our plots.

phyloseq also relies on `vegan` (ecological distances, ordination, and statistics) and `ape` (phylogenetic trees). These were installed along with phyloseq. We'll call one vegan function directly later.

In [ ]:
# Load packages -----------------------------------------------------------
library(phyloseq)
library(ggplot2)
library(dplyr)
library(colorspace)

Later on, we'll be plotting our data. Set a consistent plotting theme for all our graphs.
`theme_bw()` gives a clean black-and-white background. We also set a default size for plots in this notebook.

In [ ]:
# Set plotting theme and plot size
theme_set(theme_bw())
options(repr.plot.width = 10, repr.plot.height = 6)

# Section 2: Importing your data

You've learned how to use QIIME2 to process your 16S rRNA sequencing data. QIIME2 produces several output files (*artifacts*): the feature table, a phylogenetic tree, the taxonomy, and the sample metadata.

phyloseq stores all of these together in one object. Because everything is in one place, we can easily work across the different pieces. For example, we can use information in the sample metadata to select certain samples from the ASV table.

### How the phyloseq object was made

The QIIME2 files were already combined into a phyloseq object for you, using the `qiime2R` package. For reference, this is the code that did it. **You don't need to run it.**

```r
library(qiime2R)
data_raw <- qza_to_phyloseq(features = "qiime/table-filtered.qza",
                            tree = "qiime/rooted_tree.qza",
                            taxonomy = "qiime/taxonomy.qza",
                            metadata = "metadata/demodataR.metadata.tsv")
saveRDS(data_raw, "demodata_phyloseq.rds")
```

### Download the data

The cell below downloads the saved phyloseq object (`.rds` file) from GitHub and loads it into R.

In [ ]:
# Download and load the data ----------------------------------------------
dir.create("data", showWarnings = FALSE)
download.file("https://raw.githubusercontent.com/devindrown/demodataR/main/data/demodata_phyloseq.rds",
              destfile = "data/demodata_phyloseq.rds", mode = "wb")

data_raw <- readRDS("data/demodata_phyloseq.rds")

### The sample metadata

The sample metadata is a table with one row per sample. Here is a preview of what it looks like.

| Sample Data  | Year | Type |
|--------------|------|------|
| 2017a1PCRneg | 2017 | PCR  |
| 2017a2PCRneg | 2017 | PCR  |
| 2017aDNAneg  | 2017 | DNA  |

The first column has the name of each sample. The remaining columns describe each sample. The only requirement for combining the metadata into a phyloseq object is that the sample names must match the sample names in the feature table.

**It's good practice to make sure metadata columns are in the correct format.** We convert `Year` and `Type` into *factors*, which are R's way of representing categorical data. We also create a new column, `SampleID`, from the sample names for easier plotting later.

We use the function `sample_data()` to reach into the metadata part of the phyloseq object.

In [ ]:
# Convert Year and Type into categorical factors
sample_data(data_raw)$Year <- as.factor(sample_data(data_raw)$Year)
sample_data(data_raw)$Type <- as.factor(sample_data(data_raw)$Type)

# Create a new column with the SampleIDs
sample_data(data_raw)$SampleID <- sample_names(data_raw)

Now let's look at the object. Run the cell below. You should see output like this:

```
phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 165 taxa and 12 samples ]
sample_data() Sample Data:       [ 12 samples by 4 sample variables ]
tax_table()   Taxonomy Table:    [ 165 taxa by 7 taxonomic ranks ]
phy_tree()    Phylogenetic Tree: [ 165 tips and 164 internal nodes ]
```

(There are now 4 sample variables because we just added `SampleID`.)

In [ ]:
data_raw

What are the column names of our taxonomy table? This will be helpful later.

In [ ]:
colnames(tax_table(data_raw))

**Finally, put your data in a new container.** This way, `data_raw` stays untouched if you need to start over.

In [ ]:
mydata <- data_raw

# Section 3: Taxonomic Composition Plots

**Calculate the number of reads per sample.**

In [ ]:
sample_sums(mydata)

The output should look like this (but contain more values):

| 2017a1PCRneg | 2017a2PCRneg | 2017aDNAneg | 2017b1PCRneg |
|--------------|--------------|-------------|--------------|
| 11119        | 3525         | 4170        | 44132        |

### 💭 Stop and think

Every one of these samples is a negative control, yet each has thousands of reads. Where could those reads have come from? List at least two possible sources.

**Your answer:** *(double-click to edit)*

Now, try to **plot it.**

In [ ]:
plot_bar(mydata)

**Before you move on**, show your plot to your neighbor or the instructor. What does each bar represent? How is each bar divided?

Does your plot look like this?

![Raw Bar Plot](https://raw.githubusercontent.com/devindrown/demodataR/main/images/demodata.rawbarplot.png)

## Bar plots

Your samples have different numbers of reads after quality control, so you should convert the counts to *relative abundances*. Use this command to create a new dataset:

In [ ]:
relmydata <- transform_sample_counts(mydata, function(x) 100 * x / sum(x))

This divides each ASV count by the total count for its sample and multiplies by 100. Now each sample sums to 100%, and the bars show relative abundance within a sample.

You can use phyloseq's built-in function, `plot_bar()`, to color your bar plot:

In [ ]:
plot_bar(relmydata, fill = "Class")

The plot above shows every ASV, colored by Class. This gets confusing quickly. The code below groups the ASVs at whatever taxonomic level you're interested in.

Here, we are combining at the **Phylum** level and filtering out any phylum that makes up less than 1% of a sample.

In [ ]:
relmydata_phylum <- relmydata %>%
  tax_glom(taxrank = "Phylum") %>%  # 1. Group taxa by their Phylum.
  psmelt() %>%                      # 2. Convert to a 'long' data frame suitable for ggplot.
  filter(Abundance > 1) %>%         # 3. Keep phyla that make up >1% of the abundance.
  arrange(Phylum)                   # 4. Sort the data frame by Phylum name.

Next, we plot the results.

**A note on color.** Phyla are *categories* with no natural order, so we use a **qualitative** palette: colors that differ in hue but have similar brightness, so no group looks "more" than another. *Sequential* palettes (light to dark) are for ordered values like abundance, and *diverging* palettes (two colors meeting at a midpoint) are for values above and below a center, like change from a baseline. Using one of those for categories would suggest an order that doesn't exist.

In [ ]:
# Create a qualitative color palette for the phyla.
phylum_colors <- qualitative_hcl(length(unique(relmydata_phylum$Phylum)), palette = "Set 2")

# Create the ggplot bar chart.
ggplot(relmydata_phylum, aes(x = SampleID, y = Abundance, fill = Phylum)) +
  geom_bar(stat = "identity") +                          # Creates the stacked bars
  scale_fill_manual(values = phylum_colors) +            # Applies our custom colors
  theme(axis.title.x = element_blank()) +                # Removes the x-axis title
  ylab("Relative Abundance (Phylum > 1%) \n") +          # Sets the y-axis label
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) + # Rotates sample names for readability
  ggtitle("Composition, Phylum")                         # Adds a title

Does your plot look like this?

![Phylum Bar Plot](https://raw.githubusercontent.com/devindrown/demodataR/main/images/demodata.phylumbarplot.png)

### 💭 Stop and think

We converted every sample to relative abundance, so each sample should add up to 100%. Look closely at the tops of the bars. Do they all reach 100%? If not, why not? Look back at each step of the code that made `relmydata_phylum`.

**Your answer:** *(double-click to edit)*

## Keep digging deeper into the data

Now, let's look at the **Class** and **Family** levels. The code below combines the ASVs at each of those levels.

In [ ]:
# -- Class Level Merge --
relmydata_class <- relmydata %>%
  tax_glom(taxrank = "Class") %>%
  psmelt() %>%
  filter(Abundance > 1) %>%
  arrange(Class)

# -- Family Level Merge --
relmydata_family <- relmydata %>%
  tax_glom(taxrank = "Family") %>%
  psmelt() %>%
  filter(Abundance > 1) %>%
  arrange(Family)

Now plot each level separately.

**Class:** For this example, we use a different qualitative palette for the categories. There are lots more!

In [ ]:
# Create the Class-level ggplot.
class_colors <- rainbow_hcl(length(unique(relmydata_class$Class)))

ggplot(relmydata_class, aes(x = SampleID, y = Abundance, fill = Class)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = class_colors) +
  theme(axis.title.x = element_blank()) +   # Remove x axis title
  ylab("Relative Abundance (Class > 1%) \n") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
  ggtitle("Composition, Class")

**Family:** Here, we define the colors directly in the `ggplot` command (see `scale_fill_discrete_qualitative`). This is another way to set colors.

In [ ]:
# Create the Family-level ggplot.
ggplot(relmydata_family, aes(x = SampleID, y = Abundance, fill = Family)) +
  geom_bar(stat = "identity") +
  scale_fill_discrete_qualitative(palette = "Dark 3") +
  theme(axis.title.x = element_blank()) +   # Remove x axis title
  ylab("Relative Abundance (Family > 1%) \n") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
  ggtitle("Composition, Family")

**Before you move on**, show your plot to your neighbor or the instructor. What does each bar represent? How is each bar divided?

### ✏️ Your turn

Make a bar plot at the **Order** level. Copy the Class-level code (both the merge and the plot), and change `Class` to `Order` everywhere it appears, including variable names like `relmydata_class`.

In [ ]:
# Your Order-level merge and plot here

### 💭 Stop and think

Which taxa dominate these negative controls? Do the PCR negatives and the DNA extraction negatives look similar or different? What could explain any differences?

**Your answer:** *(double-click to edit)*

# Section 4: Beta Diversity

One of the best exploratory analyses for amplicon data is an *ordination* to visualize beta diversity. Beta diversity measures how different the microbial communities are between samples. We'll use Non-Metric Multidimensional Scaling (NMDS) with Bray-Curtis dissimilarity.

phyloseq does this in two steps: first calculate the distances between all your samples, then plot them.

**Calculate the ordination.** `ordinate()` is a phyloseq function that performs the calculations. Bray-Curtis is a common distance metric based on abundance.

NMDS is an iterative method that starts from random positions, so you'll see it print its progress as it searches for the best solution. We **set a seed** first so everyone gets the same result.

In [ ]:
# Calculate the ordination.
set.seed(3)
mydata_nmds_bray <- ordinate(
  physeq = mydata,
  method = "NMDS",
  distance = "bray"
)

**Check the stress.** NMDS squeezes the differences between all samples into two dimensions. *Stress* measures how much the picture is distorted by doing that. Lower is better. A common rule of thumb:

| Stress | Interpretation |
|---|---|
| < 0.05 | Excellent |
| 0.05 – 0.1 | Good |
| 0.1 – 0.2 | Usable, but interpret with care |
| > 0.2 | Poor; the plot may be misleading |

With very few samples, you may see a warning that the stress is (nearly) zero. That means there were so few points that NMDS could arrange them almost perfectly, which is a sign of *too little data*, not a great fit.

In [ ]:
mydata_nmds_bray$stress

**Plot the ordination.** Each point represents the entire microbial community of a single sample. Points that are closer together have more similar communities. Here we color the points by the `Year` column in our metadata and use shape to represent the `Type` column.

In [ ]:
# Plot the ordination.
plot_ordination(
  physeq = mydata,
  ordination = mydata_nmds_bray,
  title = "NMDS of mydata (bray)",
  color = "Year",
  shape = "Type"
) +
  geom_point(aes(color = Year), alpha = 0.7, size = 4)

### ✏️ Your turn

Remake the ordination plot so that **color** shows `Type` and **shape** shows `Year`. Which grouping is easier to see?

In [ ]:
# Your plot here

## Phylogenetic beta diversity: UniFrac

Bray-Curtis treats every ASV as equally different from every other ASV. But two closely related *Pseudomonas* ASVs are much more similar than a *Pseudomonas* and a *Bacillus*. **UniFrac** uses the phylogenetic tree in our phyloseq object to account for this. It measures how much of the tree's branch length is shared between two communities.

* **Weighted UniFrac** accounts for the abundance of each ASV.
* **Unweighted UniFrac** only considers presence or absence.

Here we use a different ordination method, **PCoA** (Principal Coordinates Analysis). Unlike NMDS, it doesn't start from random positions, and its axes show the percentage of variation they explain.

You may see a warning about *executing %dopar% sequentially*. It just means R is doing the calculation on one processor instead of several. It's safe to ignore.

In [ ]:
# Calculate a PCoA ordination with weighted UniFrac
mydata_pcoa_wunifrac <- ordinate(
  physeq = mydata,
  method = "PCoA",
  distance = "wunifrac"
)

plot_ordination(
  physeq = mydata,
  ordination = mydata_pcoa_wunifrac,
  title = "PCoA of mydata (weighted UniFrac)",
  color = "Year",
  shape = "Type"
) +
  geom_point(aes(color = Year), alpha = 0.7, size = 4)

### ✏️ Your turn

Repeat the UniFrac ordination using **unweighted** UniFrac. Change `"wunifrac"` to `"unifrac"`, and give your new ordination object and plot title new names.

In [ ]:
# Your unweighted UniFrac ordination and plot here

### 💭 Stop and think

Compare the Bray-Curtis, weighted UniFrac, and unweighted UniFrac plots. Do the samples group the same way? If not, what does that tell you about how the communities differ (which taxa, and how abundant)?

**Your answer:** *(double-click to edit)*

## Testing for differences: PERMANOVA

Ordinations let us *see* whether groups look different. To *test* it, we use **PERMANOVA** (permutational multivariate analysis of variance), with the function `adonis2()` from the `vegan` package.

PERMANOVA asks whether samples within a group are more similar to each other than to samples in other groups. It shuffles the group labels many times (999 here) to see how often a difference this large happens by chance. In the output, look at:

* **R2**: the proportion of variation in community composition explained by the grouping.
* **Pr(>F)**: the p-value.

We write `vegan::adonis2()` to use the function without loading the whole vegan package.

In [ ]:
# Calculate the Bray-Curtis distances and get the metadata as a data frame
bray_dist <- phyloseq::distance(mydata, method = "bray")
meta <- data.frame(sample_data(mydata))

# Test whether community composition differs by Type
set.seed(3)
vegan::adonis2(bray_dist ~ Type, data = meta, permutations = 999)

**Interpret with care.** PERMANOVA can also give a significant result when groups differ in how *spread out* they are, not just in where they're centered. And with only 12 samples, there is limited power to detect differences.

### ✏️ Your turn

Run the PERMANOVA on **weighted UniFrac** distances instead. Hint: change the `method` in `phyloseq::distance()` to `"wunifrac"`.

In [ ]:
# Your PERMANOVA on weighted UniFrac here

# Section 5: Alpha Diversity

Estimating the alpha diversity of microbial communities is tricky no matter what you do. To account for uneven sequencing depth, we first **rarefy** the data, which means subsampling every sample to the same number of reads. Here, we subsample the libraries with replacement to estimate the species abundance of the real population while standardizing sampling effort.

We'll calculate two measures:

* **Richness**: the number of ASVs observed in a sample.
* **Inverse Simpson diversity**: combines richness and evenness. It's highest when there are many ASVs at similar abundances, and low when one or a few ASVs dominate.

First, find the lowest sequencing depth in the dataset. This will be our rarefaction depth.

In [ ]:
min_lib <- min(sample_sums(mydata))
min_lib

We will subsample every sample to this minimum number of reads. Because rarefaction involves random subsampling, we repeat it 100 times and average the results to get a more stable estimate.

**Set up empty matrices to store the richness and diversity estimates.**

In [ ]:
nsamp <- nsamples(mydata)
trials <- 100

# Create empty matrices to store the results of each trial.
richness <- matrix(nrow = nsamp, ncol = trials)
row.names(richness) <- sample_names(mydata)

diversity <- matrix(nrow = nsamp, ncol = trials)
row.names(diversity) <- sample_names(mydata)

**Setting a seed makes the random subsampling reproducible**, so you'll get the same answer every time you run it.

In [ ]:
set.seed(3)

Now run the loop to perform the rarefaction 100 times. This may take a little while.

In [ ]:
for (i in 1:trials) {
  # 1. Subsample the data to the minimum library size.
  r <- rarefy_even_depth(mydata, sample.size = min_lib, verbose = FALSE, replace = TRUE)

  # 2. Calculate richness (number of observed species/ASVs).
  rich <- as.numeric(as.matrix(estimate_richness(r, measures = "Observed")))
  richness[, i] <- rich

  # 3. Calculate diversity (here using the Inverse Simpson index).
  div <- as.numeric(as.matrix(estimate_richness(r, measures = "InvSimpson")))
  diversity[, i] <- div
}

Now summarize the results from the 100 trials. Calculate the mean and standard deviation per sample for observed richness and the Inverse Simpson index, and store those values in a data frame.

Notice that we name the columns `mean_val` and `sd_val`, not `mean` and `sd`. `mean` and `sd` are already the names of R functions, and reusing them for data can cause confusing errors.

In [ ]:
# Create a new data frame for the richness results (mean and standard deviation).
rich_stats <- data.frame(
  SampleID = row.names(richness),
  mean_val = apply(richness, 1, mean),
  sd_val   = apply(richness, 1, sd),
  measure  = "Richness"
)

# Create a new data frame for the diversity results.
div_stats <- data.frame(
  SampleID = row.names(diversity),
  mean_val = apply(diversity, 1, mean),
  sd_val   = apply(diversity, 1, sd),
  measure  = "Inverse Simpson"
)

**Combine the richness and diversity estimates into one data frame.**

In [ ]:
alpha <- rbind(rich_stats, div_stats)

Add the sample metadata to this data frame using the `merge()` command.

In [ ]:
s <- data.frame(sample_data(mydata))
alphadiv <- merge(alpha, s, by = "SampleID")
head(alphadiv)

**Plot the alpha diversity results for each sample, with error bars showing the standard deviation.**

In [ ]:
ggplot(alphadiv, aes(x = SampleID, y = mean_val)) +
  geom_errorbar(aes(ymin = mean_val - sd_val, ymax = mean_val + sd_val), colour = "black", width = .1) +
  geom_point(size = 2) +
  facet_wrap(~measure, ncol = 1, scales = "free") +  # Create separate panels for each metric
  ylab("Mean across 100 rarefactions") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5)) # Rotate sample labels

### Comparing groups

The plot above tells us which *sample* is most diverse. But usually the scientific question is about *groups*: does diversity differ between PCR negatives and DNA extraction negatives? A boxplot by group answers that directly. Each point is one sample.

In [ ]:
ggplot(alphadiv, aes(x = Type, y = mean_val)) +
  geom_boxplot(outlier.shape = NA) +            # Hide outlier points (we plot all points below)
  geom_jitter(aes(color = Year), width = 0.1, size = 2) +
  facet_wrap(~measure, scales = "free") +
  ylab("Mean across 100 rarefactions") +
  ggtitle("Alpha diversity by negative control type")

### 💭 Stop and think

Do PCR negatives and DNA extraction negatives differ in richness or diversity? Think about the steps each type of control goes through. Which steps could add contaminating DNA, and would you expect one type to pick up more?

**Your answer:** *(double-click to edit)*

### Quick alpha diversity plots

*phyloseq also has a built-in function for a quick alpha diversity plot.* This is useful for a quick look but offers less customization. Note that these plots use the **unrarefied** data in `mydata`, so they don't account for differences in sequencing depth the way our analysis above does.

**You will likely see a warning** saying the data has no singletons (ASVs seen exactly once) and that this is suspicious. Here's why it appears:

* Some richness estimators, like Chao1, use the number of singletons to guess how many rare ASVs were missed. With no singletons, those estimates are unreliable.
* QIIME2's denoising step (DADA2) removes singletons on purpose, because a sequence seen only once is hard to tell apart from a sequencing error. So their absence is expected for our data, not a sign that something went wrong.
* The measures we use here (Inverse Simpson and Shannon) don't depend on singletons, so the warning doesn't affect these plots.

In [ ]:
plot_richness(mydata, measures = "InvSimpson")

There are many metrics built into phyloseq. To see them, run `help("plot_richness")` and look under `measures`. For instance, below you can plot the Shannon diversity.

In [ ]:
# Shannon Diversity
plot_richness(mydata, measures = "Shannon")

### ✏️ Your turn

`plot_richness()` can also group samples. Make a Shannon diversity plot with `Type` on the x-axis by adding `x = "Type"` to the function. Then add `+ geom_boxplot()` to the end.

In [ ]:
# Your plot here

# Wrap-up

### 💭 Stop and think

You've now looked at the composition, beta diversity, and alpha diversity of a set of negative controls. Imagine these controls were sequenced alongside real soil or gut samples.

1. How would you use what you learned here when analyzing the real samples?
2. If a taxon that's abundant in these negative controls also shows up in your real samples, what would you conclude? What else would you want to know before deciding?

**Your answer:** *(double-click to edit)*